# Configurable export of the six-dataset motor-imagery cohort


**One environment-specific consideration, upfront:** Google Drive is mounted through
a FUSE filesystem, not a real local disk. Two consequences worth knowing before a
long unattended run:

1. **Random-access reads are slower than local disk.** MOABB/MNE readers do seek
   operations while parsing EDF/MAT/FIF files, which is where FUSE overhead is most
   noticeable. `MIRROR_RAW_DATA_LOCALLY` (below) controls this: `True` copies each
   dataset's files to the Colab VM's local disk the first time they're touched
   (`/content/mne_data`), then reads from there for every subsequent subject --
   trading ~one-time copy cost for much faster repeated access. `False` reads
   directly from Drive every time, which is closer to a literal "process the data
   in place on Drive" but slower and more prone to (2).
2. **Frequent small writes to Drive intermittently fail with transient I/O errors**
   -- a known FUSE behavior, not a bug in this notebook. Every write in Sections
   5-8 (shards, checkpoint, metadata, QC report, splits, manifest) goes through
   `_write_with_retry`, defined below, which retries with exponential backoff. This
   is the Colab-specific reason that function exists at all (on Kaggle's local
   disk, in the other version of this notebook, it is a harmless pass-through).

In [ ]:
import os, sys, shutil, stat, json, hashlib, gc, time, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict, fields, is_dataclass
from typing import Any, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


In [ ]:
from zipfile import ZipFile

with ZipFile("/content/preprocessing_pkg.zip", 'r') as zObject:
    zObject.extractall(path="/content")

### Edit these paths, then run the rest of the notebook top to bottom

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PKG_SOURCE_DIR = Path("/content/preprocessing_pkg")
RAW_DATA_DRIVE_ROOT = Path("/content/drive/MyDrive/MI_EEG_Datasets")
OUTPUT_DRIVE_ROOT = Path("/content/drive/MyDrive/MI_EEG_Datasets/processed_data")

# See the Section 0 markdown above for what this trades off.
MIRROR_RAW_DATA_LOCALLY = False

for p, desc in [(PKG_SOURCE_DIR, "PKG_SOURCE_DIR"), (RAW_DATA_DRIVE_ROOT, "RAW_DATA_DRIVE_ROOT")]:
    assert p.exists(), f"{desc} does not exist: {p} -- fix the path above."
OUTPUT_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive mounted. Paths:")
print(" PKG_SOURCE_DIR      :", PKG_SOURCE_DIR)
print(" RAW_DATA_DRIVE_ROOT  :", RAW_DATA_DRIVE_ROOT)
print(" OUTPUT_DRIVE_ROOT    :", OUTPUT_DRIVE_ROOT)
print(" MIRROR_RAW_DATA_LOCALLY:", MIRROR_RAW_DATA_LOCALLY)


Drive mounted. Paths:
 PKG_SOURCE_DIR      : /content/preprocessing_pkg
 RAW_DATA_DRIVE_ROOT  : /content/drive/MyDrive/MI_EEG_Datasets
 OUTPUT_DRIVE_ROOT    : /content/drive/MyDrive/MI_EEG_Datasets/processed_data
 MIRROR_RAW_DATA_LOCALLY: False


In [ ]:
!pip install -q mne moabb pyriemann autoreject umap-learn mne-icalabel python-picard psutil pyyaml


In [ ]:
import mne
mne.set_log_level("ERROR")

# Import preprocessing_pkg SUBMODULES
if str(PKG_SOURCE_DIR.parent) not in sys.path:
    sys.path.insert(0, str(PKG_SOURCE_DIR.parent))
package_import_name = PKG_SOURCE_DIR.name   # must be "preprocessing_pkg" for the imports below to resolve
assert package_import_name == "preprocessing_pkg", (
    f"PKG_SOURCE_DIR's folder is named {package_import_name!r}; rename it to "
    f"'preprocessing_pkg' or symlink it, since the imports below assume that name.")

import preprocessing_pkg.config as pkgconfig
import preprocessing_pkg.channels as pkgchannels
import preprocessing_pkg.io_loaders as pkgio
import preprocessing_pkg.signal_processing as pkgsp
import preprocessing_pkg.pipeline_builder as pkgpb
import preprocessing_pkg.alignment as pkgalign
import preprocessing_pkg.audits as pkgaudits
import preprocessing_pkg.cho2017_qc as pkgcho

import moabb
print("preprocessing_pkg submodules imported directly (no torch/pyriemann dependency).")
print("mne", mne.__version__, "| moabb", moabb.__version__)


preprocessing_pkg submodules imported directly (no torch/pyriemann dependency).
mne 1.13.2 | moabb 1.7.2


In [ ]:
# --- Point MNE/MOABB at the raw-data cache -------------------------------------------
SIGNAL_EXTENSIONS = {'.edf', '.mat', '.fif', '.set', '.fdt', '.vhdr', '.vmrk', '.eeg', '.pdf', '.bdf', '.gdf'}

drive_mne_data_candidates = list(RAW_DATA_DRIVE_ROOT.rglob("mne_data"))
assert drive_mne_data_candidates, f"No 'mne_data' folder found under {RAW_DATA_DRIVE_ROOT}."
DRIVE_MNE_DATA_DIR = drive_mne_data_candidates[0]

if MIRROR_RAW_DATA_LOCALLY:
    # Local mirror on the Colab VM's own disk, built once. Small metadata/index files
    # are copied outright; large signal files are SYMLINKED (not copied) back to
    # Drive -- this keeps the one-time mirror step fast and avoids doubling Drive
    # storage use, while still giving MNE/MOABB fast local directory listings and
    # metadata access. The bytes of large recordings are still ultimately read from
    # Drive on first access, but MNE's own on-disk cache (Section below) then keeps
    # anything it touches locally for every subsequent subject.
    MNE_DATA_DIR = Path('/content/mne_data')
    MNE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    n_copied = n_linked = n_skipped = 0
    for root, dirs, files in os.walk(DRIVE_MNE_DATA_DIR):
        rel_path = Path(root).relative_to(DRIVE_MNE_DATA_DIR)
        dest_dir = MNE_DATA_DIR / rel_path
        dest_dir.mkdir(parents=True, exist_ok=True)
        for filename in files:
            src_file, dest_file = Path(root) / filename, dest_dir / filename
            if dest_file.exists() or dest_file.is_symlink():
                n_skipped += 1
                continue
            if src_file.suffix.lower() in SIGNAL_EXTENSIONS:
                try:
                    dest_file.symlink_to(src_file)
                    n_linked += 1
                except FileExistsError:
                    pass
            else:
                shutil.copyfile(src_file, dest_file)
                os.chmod(dest_file, dest_file.stat().st_mode | stat.S_IWRITE)
                n_copied += 1
    print(f"Local mirror built at {MNE_DATA_DIR}: {n_copied} small files copied, "
          f"{n_linked} large files symlinked, {n_skipped} already present.")
else:
    MNE_DATA_DIR = DRIVE_MNE_DATA_DIR
    print(f"Reading raw data directly from Drive (no local mirror): {MNE_DATA_DIR}")

MNE_DATA_DIR_STR = str(MNE_DATA_DIR)
PHYSIONET_DIR = os.path.join(MNE_DATA_DIR_STR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0')
mne.set_config('MNE_DATA', MNE_DATA_DIR_STR)
mne.set_config('MNE_DATASETS_EEGBCI_PATH', MNE_DATA_DIR_STR)
moabb.set_download_dir(MNE_DATA_DIR_STR)
os.environ['PHYSIONET_ROOT'] = PHYSIONET_DIR
print("MNE_DATA:", MNE_DATA_DIR_STR)


Reading raw data directly from Drive (no local mirror): /content/drive/MyDrive/MI_EEG_Datasets/mne_data
MNE_DATA: /content/drive/MyDrive/MI_EEG_Datasets/mne_data


In [ ]:
# Pipeline's own on-disk cache (filter design cache etc.) -- always LOCAL regardless
# of MIRROR_RAW_DATA_LOCALLY, since these are small, ephemeral, and read/written
# extremely frequently (exactly the write pattern most likely to hit transient Drive
# FUSE errors, and least valuable to persist across sessions anyway).
OUTPUT_ROOT = OUTPUT_DRIVE_ROOT   # Sections 3+ write shards/metadata/QC/etc. here, on Drive
CACHE_DIR = "/content/pipeline_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
pkgsp.configure_cache(CACHE_DIR)
DEFAULT_OUTPUT_ROOT = str(OUTPUT_DRIVE_ROOT)   # Section 1's ExportConfig reads this
print("Output root (on Drive):", OUTPUT_ROOT)
print("Local scratch cache:", CACHE_DIR)


[cache] Disk cache enabled at: /content/pipeline_cache
Output root (on Drive): /content/drive/MyDrive/MI_EEG_Datasets/processed_data
Local scratch cache: /content/pipeline_cache


In [ ]:
# Every write in Sections 5-8 goes through this. Drive's FUSE layer intermittently
# raises transient OSErrors ("Input/output error", "Transport endpoint is not
# connected") under frequent small writes -- this retries with exponential backoff
# before giving up. See the Section 0 markdown for context.
def _write_with_retry(fn, *args, retries=6, initial_delay=2.0, **kwargs):
    delay = initial_delay
    last_exc = None
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except OSError as exc:
            last_exc = exc
            print(f"  [Drive IO retry {attempt+1}/{retries}] {type(exc).__name__}: {exc}; "
                  f"retrying in {delay:.1f}s")
            time.sleep(delay)
            delay *= 2
    raise RuntimeError(
        f"Write failed after {retries} attempts against Drive -- this usually means "
        f"more than a transient hiccup (session re-auth needed, quota, or a broken "
        f"path). Re-run the drive.mount() cell above and try again."
    ) from last_exc


## Section 1 — Full configuration

Every value below can be changed. Two groups, deliberately kept separate:

- **`PreprocConfig`** — anything that changes the *numbers written to disk*
  (sampling rate, band-pass, harmonization mode, CAR scope, artifact variant, the
  master epoch window). Changing any of these fields changes `preproc_id()`, a short
  hash stamped into every output file, so two exports with different settings can
  never be silently mixed.
- **Everything else** (`DatasetSelectConfig`, `RamConfig`, `SplitConfig`, `QCConfig`)
  — knobs that do not change what is stored, only which subjects are included, how
  the export loop schedules work, how splits are drawn, and what QC treats as a
  pass/fail threshold.

**Deliberately NOT configurable here: normalization and Euclidean Alignment are never
baked into the stored arrays.** Both are statistics fit on a set of trials, and which
trials are "training" trials depends on the split -- a split this notebook does not
yet know the final use of (Stage 0 is still open). Baking either in at export time
would silently make every future split's "held-out" subjects touch statistics from
themselves. Instead: raw physical units (uV, float32) are stored, and this notebook's
own leakage validation (Section 8) proves that per-split normalization is exactly the
kind of thing that must remain a *load-time*, per-split operation. `EA_ENABLED` below
only controls whether a per-subject spatial-covariance reference is *computed and
stored as metadata* for optional later use -- it is never applied to `X`.

In [ ]:
def _canon(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=list)


@dataclass(frozen=True)
class DatasetSelectConfig:
    '''Per-dataset knobs that decide WHICH raw data is used. None of these change
    the numbers computed FROM a trial, only which trials/subjects are included.'''
    enabled: bool = True
    subjects: Any = "all"                 # "all" | int (first N) | explicit list[int]
    extra_excluded_subjects: tuple = ()    # on top of the package's documented exclusions
    lee2019_sessions: tuple = (1, 2)       # BOTH offline sessions by default (required)
    dreyer2023_acquisition_only: bool = False
    # False (default): keep all 6 runs. Run-level provenance (Section 4) records which
    # run every trial came from, so the feedback-confound runs (R3-R6, feedback from
    # cue+1.25s) can be filtered out at LOAD time instead of discarded at export time.
    # Set True to reproduce the main notebook's documented acquisition-only default.
    physionet_allow_anomalous: bool = False


@dataclass(frozen=True)
class PreprocConfig:
    '''Everything that changes the numbers written into the stored arrays.
    Hashed into preproc_id(); changing any field invalidates old exports.'''
    version: str = "v1"
    target_sfreq: float = 128.0
    bandpass: tuple = (8.0, 30.0)
    fir_trans_bandwidth: float = 2.0
    montage_name: str = "standard_1005"
    harmonization_mode: str = "reference_masked"   # "reference_masked" | "union" | "intersection"
    anchor_datasets: tuple = ("physionet", "cho2017")
    car_scope: str = "shared_support"               # "shared_support" | "native"
    artifact_variant: str = "primary"                # "primary" | "ica" | "autoreject" | "channel_reject"
    # Cue-relative [tmin, tmax) of the MASTER epoch stored on disk. Wider than any one
    # analysis window on purpose: a window ablation becomes a load-time CROP of this,
    # never a re-export. Validated against every enabled dataset's own trial_length_sec
    # / pre_cue_clean_sec in Section 3 before export starts.
    master_window: tuple = (0.0, 3.0)
    units: str = "uV"
    dtype: str = "float32"
    label_map: dict = field(default_factory=lambda: {"left_hand": 0, "right_hand": 1})
    random_state: int = 42
    ea_enabled: bool = True     # compute + store per-subject EA reference as METADATA only
    ea_scope: str = "subject"   # documents which scope the stored reference matches

    def __post_init__(self):
        if self.dtype != "float32":
            raise ValueError("Store float32 -- values are in uV; float16 risks over/underflow.")
        if self.units != "uV":
            raise ValueError("units must be 'uV' (converted from MNE's native Volts at export).")
        span = (self.master_window[1] - self.master_window[0]) * self.target_sfreq
        if abs(span - round(span)) > 1e-6:
            raise ValueError(f"master_window does not give an integer sample count ({span}).")

    @property
    def n_samples(self) -> int:
        return int(round((self.master_window[1] - self.master_window[0]) * self.target_sfreq))

    def preproc_id(self) -> str:
        return hashlib.sha1(_canon(asdict(self)).encode()).hexdigest()[:12]


@dataclass(frozen=True)
class RamConfig:
    '''Export-time performance / memory-safety knobs. Subjects are processed in
    parallel (n_jobs chosen from live free RAM, see Section 5) and each one writes
    immediately to disk; this controls how that loop schedules work, never whether
    results depend on it -- output is identical regardless of n_jobs or chunking.'''
    n_jobs: int = 1                    # sequential by default
    free_ram_reserve_gb: float = 2.0
    per_worker_gb_budget: float = 1.5
    resume: bool = True                # skip subjects whose shard already exists + verifies
    gc_every_subject: bool = True
    lee2019_process_sessions_separately: bool = True  # halves peak RAM for the one native-1000Hz dataset


@dataclass(frozen=True)
class SplitConfig:
    '''Subject-level, leakage-free split manifest parameters (Section 8).'''
    protocol: str = "pooled_group_kfold"
    n_folds: int = 5
    val_frac: float = 0.15
    locked_test_frac: float = 0.15
    stratify_by_dataset: bool = True
    seed: int = 42


@dataclass(frozen=True)
class QCConfig:
    '''Thresholds the QC gate checks against (Section 7). Exceeding these does not
    crash the export -- it marks qc_status='fail'/'warn' per subject and dataset, and
    a 'fail' anywhere blocks the final READY_FOR_TRAINING status.'''
    amplitude_median_abs_uv_min: float = 0.5
    amplitude_median_abs_uv_max: float = 500.0
    max_nan_fraction: float = 0.0
    min_trials_per_class: int = 5
    max_class_imbalance_ratio: float = 3.0
    require_masked_exactly_zero: bool = True
    require_deterministic_rebuild: bool = True
    n_determinism_check_subjects: int = 3


@dataclass(frozen=True)
class ExportConfig:
    run_name: str = "eeg_export_v1"
    output_root: str = field(default_factory=lambda: DEFAULT_OUTPUT_ROOT)
    seed: int = 42
    datasets: dict = field(default_factory=lambda: {
        "physionet": DatasetSelectConfig(),
        "cho2017": DatasetSelectConfig(),
        "lee2019": DatasetSelectConfig(),
        "bnci2014001": DatasetSelectConfig(),
        "weibo2014": DatasetSelectConfig(),
        "dreyer2023": DatasetSelectConfig(),
    })
    preproc: PreprocConfig = field(default_factory=PreprocConfig)
    ram: RamConfig = field(default_factory=RamConfig)
    split: SplitConfig = field(default_factory=SplitConfig)
    qc: QCConfig = field(default_factory=QCConfig)

    def enabled_dataset_names(self) -> list:
        return [k for k, v in self.datasets.items() if v.enabled]

    def to_jsonable(self) -> dict:
        def conv(o):
            if is_dataclass(o):
                return {k: conv(v) for k, v in asdict(o).items()}
            if isinstance(o, dict):
                return {k: conv(v) for k, v in o.items()}
            if isinstance(o, (list, tuple)):
                return [conv(v) for v in o]
            return o
        return conv(self)


### Edit configuration

Every field has a default matching the main notebook's primary pipeline; override only what you want to change.
Examples of common overrides are commented in place.

In [ ]:
CFG = ExportConfig(
    run_name="eeg_export_v2",
    output_root=DEFAULT_OUTPUT_ROOT,
    seed=42,

    datasets={
        "physionet":   DatasetSelectConfig(subjects="all"),
        "cho2017":     DatasetSelectConfig(subjects="all"),
        "lee2019":     DatasetSelectConfig(subjects="all", lee2019_sessions=(1, 2)),   # BOTH sessions
        "bnci2014001": DatasetSelectConfig(subjects="all"),
        "weibo2014":   DatasetSelectConfig(subjects="all"),
        "dreyer2023":  DatasetSelectConfig(subjects="all", dreyer2023_acquisition_only=False),
    },

    preproc=PreprocConfig(
        target_sfreq=128.0,
        bandpass=(8.0, 32.0),               # <- analysis band, e.g. (4.0, 40.0)
        fir_trans_bandwidth=2.0,
        montage_name="standard_1005",
        harmonization_mode="reference_masked",   # <- "union" | "intersection" | "reference_masked"
        anchor_datasets=("physionet", "cho2017"),
        car_scope="shared_support",
        artifact_variant="primary",          # <-  "primary" | "ica" | "autoreject" | "channel_reject"
        master_window=(0.0, 3.0),            # <- wide master epoch; crop at load time
        ea_enabled=True,                     # <- compute+store EA reference as metadata only
        ea_scope="subject",
    ),

    ram=RamConfig(
        n_jobs=2,                            # sequential export; see Section 6 for why
        resume=True,
        lee2019_process_sessions_separately=True,
    ),

    split=SplitConfig(
        n_folds=5,
        val_frac=0.15,
        locked_test_frac=0.15,
        seed=42,
    ),

    qc=QCConfig(),
)

print(f"preproc_id = {CFG.preproc.preproc_id()}   (stamped into every output file)")
print(f"master epoch: {CFG.preproc.master_window} s cue-relative, "
      f"{CFG.preproc.n_samples} samples @ {CFG.preproc.target_sfreq} Hz")
print(f"enabled datasets: {CFG.enabled_dataset_names()}")


preproc_id = a46230154bfe   (stamped into every output file)
master epoch: (0.0, 3.0) s cue-relative, 384 samples @ 128.0 Hz
enabled datasets: ['physionet', 'cho2017', 'lee2019', 'bnci2014001', 'weibo2014', 'dreyer2023']


In [ ]:
# Print the full, resolved configuration as one table -- nothing hidden in a cell below.
def _flatten_cfg(prefix, obj, rows):
    if is_dataclass(obj):
        for f in fields(obj):
            _flatten_cfg(f"{prefix}{f.name}.", getattr(obj, f.name), rows)
    elif isinstance(obj, dict):
        for k, v in obj.items():
            _flatten_cfg(f"{prefix}{k}.", v, rows)
    else:
        rows.append((prefix.rstrip("."), obj))

_rows = []
_flatten_cfg("", CFG, _rows)
cfg_table = pd.DataFrame(_rows, columns=["parameter", "value"])
pd.set_option("display.max_rows", 200)
display(cfg_table)


,parameter,value
0,run_name,eeg_export_v2
1,output_root,/content/drive/MyDrive/MI_EEG_Datasets/process...
2,seed,42
3,datasets.physionet.enabled,True
4,datasets.physionet.subjects,all
5,datasets.physionet.extra_excluded_subjects,()
6,datasets.physionet.lee2019_sessions,"(1, 2)"
7,datasets.physionet.dreyer2023_acquisition_only,False
8,datasets.physionet.physionet_allow_anomalous,False
9,datasets.cho2017.enabled,True


## Section 2 — Dataset registry, with provenance-preserving loaders

`DatasetConfig.loader_fn(subject) -> list[Raw]` is the package's loader contract, used
unchanged. The wrappers below call the *exact same* MOABB dataset classes and the
*exact same* selection logic as `preprocessing_pkg.io_loaders` (Lee2019's session
selection, Dreyer2023's `acquisition_only` run filter) -- they change nothing about
which samples are loaded or how. The one thing they add: MOABB's own loaders discard
each run's session/run key after picking channels, so the package's `loader_fn`
contract has no way to tell two runs of one subject apart. Downstream (Section 4) we
need that to build honest per-trial metadata (dataset identity is already discoverable
from the channel mask alone -- run/session identity would otherwise be lost entirely).
Each wrapper stamps `\"session=<s>;run=<r>\"` into `raw.info['description']`, MNE's own
free-form text field, before returning -- nothing else about the Raw is touched.

**This registry is built once, correctly, and never rebuilt.** The main notebook has an
open bug (Stage 0, not yet fixed) where a later cell silently rebuilds the registry with
unbound loaders, undoing the Lee2019-session and Dreyer2023-acquisition-only choices
made earlier. There is exactly one registry-construction cell below.

In [ ]:
def _tag_provenance(raw, session, run):
    raw.info["description"] = f"session={session};run={run}"
    return raw


def _parse_provenance(raw):
    desc = raw.info.get("description") or ""
    session, run = "0", "0"
    for part in desc.split(";"):
        if part.startswith("session="):
            session = part[len("session="):]
        elif part.startswith("run="):
            run = part[len("run="):]
    return session, run


def _moabb_all_runs_with_provenance(dataset, subject: int, run_filter=None) -> list:
    '''Same selection logic as preprocessing_pkg.io_loaders._moabb_all_runs, plus
    provenance tagging. subject/session/run keys come straight from MOABB's own
    get_data() dict, so this is the ground truth, not a guess.'''
    data = dataset.get_data(subjects=[subject])[subject]
    raws = []
    for s in data:
        for r in data[s]:
            if run_filter is not None and not run_filter(str(r)):
                continue
            raw = data[s][r].copy().pick(picks="eeg")
            raws.append(_tag_provenance(raw, s, r))
    if not raws:
        raise RuntimeError(f"No runs survived the run filter for subject {subject}.")
    return raws


def make_physionet_loader_with_provenance():
    def _loader(subject: int) -> list:
        raws = pkgio.load_physionet_subject_runs(subject, allow_anomalous=CFG.datasets["physionet"].physionet_allow_anomalous)
        for raw, run in zip(raws, pkgconfig.PHYSIONET_UNILATERAL_IMAGERY_RUNS):
            _tag_provenance(raw, session="1", run=str(run))
        return raws
    return _loader


def make_cho2017_loader_with_provenance():
    from moabb.datasets import Cho2017
    def _loader(subject: int) -> list:
        return _moabb_all_runs_with_provenance(Cho2017(), subject)
    return _loader


def make_lee2019_loader_with_provenance(sessions):
    def _loader(subject: int) -> list:
        from moabb.datasets import Lee2019_MI
        dataset = Lee2019_MI(train_run=True, test_run=False, sessions=list(sessions))
        data = dataset.get_data(subjects=[subject], cache_config=dict(save_raw=True))[subject]
        if not data:
            raise RuntimeError(f"Lee2019 subject {subject}: get_data() returned no sessions.")
        raws = []
        for s in data:
            for r in data[s]:
                raw = pkgio._lee2019_attach_annotations(data[s][r]).pick(picks="eeg")
                raws.append(_tag_provenance(raw, session=str(s), run=str(r)))
        return raws
    return _loader


def make_bnci2014001_loader_with_provenance():
    from moabb.datasets import BNCI2014_001
    def _loader(subject: int) -> list:
        return _moabb_all_runs_with_provenance(BNCI2014_001(), subject)
    return _loader


def make_weibo2014_loader_with_provenance():
    from moabb.datasets import Weibo2014
    def _loader(subject: int) -> list:
        return _moabb_all_runs_with_provenance(Weibo2014(), subject)
    return _loader


def make_dreyer2023_loader_with_provenance(acquisition_only: bool):
    from moabb.datasets import Dreyer2023
    run_filter = None
    if acquisition_only:
        def run_filter(name: str) -> bool:
            return ("R1" in name) or ("R2" in name)
    def _loader(subject: int) -> list:
        return _moabb_all_runs_with_provenance(Dreyer2023(), subject, run_filter=run_filter)
    return _loader


print("Provenance-preserving loader factories defined (same MOABB calls, same selection logic).")


Provenance-preserving loader factories defined (same MOABB calls, same selection logic).


In [ ]:
# --- Channel-info probes (cached: each opens/downloads a recording once) -------------
_channel_info_cache = {}

def _cached_channel_info(name, probe_fn):
    def _fn():
        if name not in _channel_info_cache:
            names, types, sfreq = probe_fn()
            from preprocessing_pkg.channels import scalp_eeg_names
            _channel_info_cache[name] = scalp_eeg_names(names, types)
            print(f"[{name}] {len(_channel_info_cache[name])} scalp EEG channels @ {sfreq:.0f} Hz")
        return _channel_info_cache[name]
    return _fn


registry = pkgconfig.DatasetRegistry(
    pkgconfig.make_physionet_config(
        make_physionet_loader_with_provenance(),
        _cached_channel_info("physionet", pkgio.get_physionet_channel_info)),
    pkgconfig.make_cho2017_config(
        make_cho2017_loader_with_provenance(),
        _cached_channel_info("cho2017", pkgio.get_cho2017_channel_info)),
    pkgconfig.make_lee2019_config(
        make_lee2019_loader_with_provenance(CFG.datasets["lee2019"].lee2019_sessions),
        _cached_channel_info("lee2019", pkgio.get_lee2019_channel_info)),
    pkgconfig.make_bnci2014001_config(
        make_bnci2014001_loader_with_provenance(),
        _cached_channel_info("bnci2014001", pkgio.get_bnci2014001_channel_info)),
    pkgconfig.make_weibo2014_config(
        make_weibo2014_loader_with_provenance(),
        _cached_channel_info("weibo2014", pkgio.get_weibo2014_channel_info)),
    pkgconfig.make_dreyer2023_config(
        make_dreyer2023_loader_with_provenance(CFG.datasets["dreyer2023"].dreyer2023_acquisition_only),
        _cached_channel_info("dreyer2023", pkgio.get_dreyer2023_channel_info)),
)
registry.enable_only(*CFG.enabled_dataset_names())

for cfg in registry.enabled():
    select = CFG.datasets[cfg.name]
    excluded = dict(cfg.excluded_subjects)
    if cfg.name == "physionet" and select.physionet_allow_anomalous:
        # cfg.excluded_subjects defaults to PHYSIONET_KNOWN_ANOMALOUS_SUBJECTS (88/89/92/100).
        # Opting in removes them here AND the provenance loader was built with
        # allow_anomalous=True (Section 2), so the loader will not raise for them.
        removed = {sid: reason for sid, reason in excluded.items()}
        excluded = {}
        print(f"  [physionet] allow_anomalous=True: including {list(removed)} "
              f"(documented anomalies: {removed})")
    extra = set(select.extra_excluded_subjects)
    excluded.update({sid: "excluded via ExportConfig.extra_excluded_subjects" for sid in extra})
    cfg.excluded_subjects = excluded

print("Registry built ONCE. Configured:", [c.name for c in registry.enabled()])


Registry built ONCE. Configured: ['physionet', 'cho2017', 'lee2019', 'bnci2014001', 'weibo2014', 'dreyer2023']


In [ ]:
# --- Cue-onset resolution -- same package call as the main notebook -----------------
print("MOABB places annotations at onset+interval[0]:", pkgio.moabb_annotations_include_interval_offset())
offsets = pkgio.apply_cue_offsets(registry)
_ = registry.cue_alignment_table()


MOABB places annotations at onset+interval[0]: True
[cue offset] physionet: +0.00s (annotations built directly at cue onset)
[cue offset] cho2017: +0.00s (MOABB interval starts at 0)
[cue offset] lee2019: +0.00s (annotations built directly at cue onset)
[cue offset] bnci2014001: +0.00s -- MOABB 1.7.2 >= 1.1 already places annotations at onset+interval[0] (2.0s).
[cue offset] weibo2014: +0.00s -- MOABB 1.7.2 >= 1.1 already places annotations at onset+interval[0] (3.0s).
[cue offset] dreyer2023: +0.00s (MOABB interval starts at 0)

================= CUE-ONSET ALIGNMENT =================
    dataset moabb_interval  cue_offset_sec  post-cue trial length (s)  clean pre-cue (s)  feedback onset (s post-cue)
  physionet     (0.0, 3.0)             0.0                        4.0                2.0                          NaN
    cho2017     (0.0, 3.0)             0.0                        3.0                2.0                          NaN
    lee2019     (0.0, 4.0)             0.0            

In [ ]:
# --- Channel harmonization -- same package call as the main notebook ----------------
channel_dfs = []
for cfg in registry.enabled():
    names = cfg.get_channel_names()
    channel_dfs.append(pkgchannels.filter_eeg_only(
        pkgchannels.build_channel_table(names, ["eeg"] * len(names), cfg.name)))

common_channels, missing_per_dataset = pkgchannels.resolve_target_channels(
    *channel_dfs, mode=CFG.preproc.harmonization_mode,
    anchor_datasets=list(CFG.preproc.anchor_datasets))

channel_mapping_df = pkgchannels.build_mapping_table(*channel_dfs, common_channels=common_channels)
channel_masks = pkgchannels.build_channel_masks(*channel_dfs, target_channels=common_channels)

montage, resolved_positions, missing_montage = pkgchannels.verify_common_channels_montage(
    common_channels, montage_name=CFG.preproc.montage_name)
assert not missing_montage, f"Unresolved channel(s): {missing_montage}"
assert common_channels == sorted(common_channels)

print(f"SHARED REPRESENTATION: {len(common_channels)} channels, mode='{CFG.preproc.harmonization_mode}'")
for name, m in channel_masks.items():
    print(f"  {name:13s} {int(m.sum()):3d}/{len(m)} real  ({100*m.mean():5.1f}% coverage)")


[physionet] 64 scalp EEG channels @ 160 Hz


[cho2017] 64 scalp EEG channels @ 512 Hz
[lee2019] 62 scalp EEG channels @ 1000 Hz
[bnci2014001] 22 scalp EEG channels @ 250 Hz


[weibo2014] 60 scalp EEG channels @ 200 Hz


9it [00:00, 918.53it/s]


[dreyer2023] 27 scalp EEG channels @ 512 Hz
Montage 'standard_1005': 62/62 target channels resolved to a known scalp position.
SHARED REPRESENTATION: 62 channels, mode='reference_masked'
  physionet      62/62 real  (100.0% coverage)
  cho2017        62/62 real  (100.0% coverage)
  lee2019        48/62 real  ( 77.4% coverage)
  bnci2014001    22/62 real  ( 35.5% coverage)
  weibo2014      58/62 real  ( 93.5% coverage)
  dreyer2023     27/62 real  ( 43.5% coverage)


In [ ]:
# --- CAR scope -- same package call as the main notebook ----------------------------
from types import SimpleNamespace
car_scope = pkgpb.resolve_car_scope(registry, common_channels,
                                     SimpleNamespace(CAR_SCOPE=CFG.preproc.car_scope))
print(f"CAR scope ({CFG.preproc.car_scope}): {len(car_scope) if car_scope else 'native (per-dataset)'} channel(s)")


[CAR scope] 'shared_support': averaging over the 18 channel(s) every enabled dataset recorded (of 62 target channels).
CAR scope (shared_support): 18 channel(s)


In [ ]:
# --- Master-window validation, per dataset, against ITS OWN trial length / pre-cue budget
_problems = []
for cfg in registry.enabled():
    tmin, tmax = CFG.preproc.master_window
    if tmax > cfg.trial_length_sec + 1e-9:
        _problems.append(f"{cfg.name}: master_window tmax={tmax:.3f}s exceeds its "
                          f"{cfg.trial_length_sec}s post-cue trial length.")
    if tmin < -cfg.pre_cue_clean_sec - 1e-9:
        _problems.append(f"{cfg.name}: master_window tmin={tmin:.3f}s reaches further back "
                          f"than its {cfg.pre_cue_clean_sec}s of clean pre-cue signal.")
if _problems:
    raise AssertionError("Master-window validation failed:\n" + "\n".join(_problems))
print(f"master_window {CFG.preproc.master_window} validated against every enabled dataset's "
      f"own trial length and pre-cue budget.")

for cfg in registry.enabled():
    if cfg.feedback_onset_sec is not None and CFG.preproc.master_window[1] > cfg.feedback_onset_sec:
        print(f"NOTE (not blocking): {cfg.name} master_window extends past its "
              f"{cfg.feedback_onset_sec}s online-feedback onset. The run-level provenance "
              f"stored in Section 4 lets you exclude the affected runs at load time instead "
              f"of losing them here.")


master_window (0.0, 3.0) validated against every enabled dataset's own trial length and pre-cue budget.
NOTE (not blocking): dreyer2023 master_window extends past its 1.25s online-feedback onset. The run-level provenance stored in Section 4 lets you exclude the affected runs at load time instead of losing them here.


## Section 3 — Metadata schema

Three tables, one row per unit of interest. Every field is populated during export as
a side effect of processing (never reconstructed afterwards from the arrays alone).

**`trials` (one row per trial -- the atomic unit):**

| field | meaning |
|---|---|
| `trial_uid` | globally unique string key |
| `dataset`, `dataset_idx` | dataset name and a stable integer code |
| `subject_id` | dataset-local subject number |
| `global_subject_idx` | unique subject index across all 6 datasets (for subject-ID heads) |
| `session` | session key as reported by the loader (Lee2019: `1`/`2`; single-session sets: `1`) |
| `run_label` | run/phase key as reported by the loader (PhysioNet: `4`/`8`/`12`; Dreyer2023: `R1`..`R6`; ...) |
| `run_index` | 0-based position of this run within the subject's loaded run list |
| `trial_index_in_run` | 0-based trial position within its run |
| `global_trial_index` | 0-based position within this subject's full trial set |
| `label_str`, `label_int` | `left_hand`/`right_hand`, and `0`/`1` per `PreprocConfig.label_map` |
| `event_sample_master` | sample index of the cue inside the stored master-window array |
| `orig_event_sample` | absolute sample index of the matched annotation before cropping (debug/traceability) |
| `n_real_channels` | channels that are real (not zero-filled) for this trial's dataset |
| `native_sfreq` | this dataset's native sampling rate before resampling |
| `cue_offset_sec`, `feedback_onset_sec`, `pre_cue_clean_sec` | per-dataset timing constants, repeated per row for convenience |
| `preproc_id` | the config hash this trial was produced under |
| `shard_path`, `row_in_shard` | which `.npz` file holds this trial's array, and at what index |

**`subjects` (one row per exported (dataset, subject)):**
`dataset`, `subject_id`, `global_subject_idx`, `n_trials`, `n_trials_left`, `n_trials_right`,
`n_runs`, `run_labels`, `n_real_channels`, `native_sfreq`, `shard_path`, `shard_sha1`,
`shard_shape`, `ea_reference_stored` (bool), `qc_status`, `qc_notes`, `processing_seconds`,
`export_timestamp`, `package_versions`.

**`datasets_summary` (one row per dataset):**
`dataset`, `n_subjects_in_registry`, `n_excluded_by_config`, `n_exported`, `n_failed`,
`failed_subjects` (id -> reason), `n_real_channels`, `n_target_channels`, `native_sfreq`.

In [ ]:
DATASET_IDX = {name: i for i, name in enumerate(sorted(CFG.enabled_dataset_names()))}
LABEL_MAP = CFG.preproc.label_map
TARGET_CHANNELS = list(common_channels)
N_SAMPLES = CFG.preproc.n_samples

PROCESSED_ROOT = Path(CFG.output_root)
DATA_ROOT = PROCESSED_ROOT / "data"
META_ROOT = PROCESSED_ROOT / "metadata"
CONFIG_ROOT = PROCESSED_ROOT / "config"
CHANNELS_ROOT = PROCESSED_ROOT / "channels"
QC_ROOT = PROCESSED_ROOT / "qc"
SPLITS_ROOT = PROCESSED_ROOT / "splits"
VALIDATION_ROOT = PROCESSED_ROOT / "validation"
for d in (DATA_ROOT, META_ROOT, CONFIG_ROOT, CHANNELS_ROOT, QC_ROOT, SPLITS_ROOT, VALIDATION_ROOT):
    d.mkdir(parents=True, exist_ok=True)

# global subject index: stable, deterministic (sorted by dataset name, then subject id),
# assigned AFTER we know which subjects actually exist to export (Section 5), so it has
# no gaps. Declared here; populated once subject lists are resolved.
GLOBAL_SUBJECT_IDX = {}   # (dataset, subject_id) -> int

print(f"Output layout created under: {PROCESSED_ROOT}")


Output layout created under: /content/drive/MyDrive/MI_EEG_Datasets/processed_data


## Section 4 — Core per-subject exporter

Mirrors `signal_processing.run_primary_pipeline` exactly: harmonize -> filter -> resample
(per run) -> concatenate -> CAR (once) -> cue-aligned epoch on the master window. The
only addition is bookkeeping: cumulative sample boundaries are recorded per run BEFORE
concatenation, so every resulting trial can be mapped back to the run it came from by
comparing its event sample against those boundaries.

In [ ]:
import psutil
from threadpoolctl import threadpool_limits

# Only the "primary" variant (harmonize -> filter -> resample -> CAR -> epoch, no ICA/
# AutoReject) is implemented by the exporter below. It is the headline pipeline the main
# notebook evaluates, and it is the only variant cheap enough to run on the full ~300
# subjects. "ica"/"autoreject"/"channel_reject" are real functions in the package
# (pkgsp.run_secondary_ica_pipeline, rejection.run_secondary_autoreject_pipeline) but are
# NOT wired into this export loop -- ICA alone took up to ~15 min/subject in the main
# notebook's own ablations, which does not scale to the full cohort. Exporting one of
# those variants for a smaller subset is a natural follow-up, not done here.
assert CFG.preproc.artifact_variant == "primary", (
    f"artifact_variant={CFG.preproc.artifact_variant!r} is not implemented by this "
    f"exporter yet -- only 'primary' is. See the comment above this assertion.")


def _harmonized_filtered_runs(raws, channel_mapping_df, common_channels, montage, glob_bandpass,
                               target_sfreq, harmonization_mode, trans_bandwidth):
    '''Per-run harmonize+filter+resample -- identical call/order to
    signal_processing._filter_resample_concat -- returning the list of processed runs
    (not yet concatenated) so their sample counts can be used as boundaries.'''
    processed = []
    for r in raws:
        pr = pkgsp.preprocess_raw(r, channel_mapping_df, common_channels, montage,
                                   l_freq=glob_bandpass[0], h_freq=glob_bandpass[1],
                                   target_sfreq=target_sfreq, mode=harmonization_mode,
                                   trans_bandwidth=trans_bandwidth)
        processed.append(pr)
    return processed


def _cho2017_qc_mask(cfg, subject, ep, event_sample_master, run_labels_per_trial):
    '''Cho2017 ships its own authors' per-trial bad-trial indices. Apply them here (same
    source pkg.cho2017_qc uses) so excluded trials never reach the stored shard at all,
    rather than being silently included and only flagged later.'''
    if cfg.name != "cho2017":
        return np.ones(len(ep), dtype=bool)
    try:
        bad_idx = pkgcho.get_cho2017_bad_trial_indices(subject)
    except Exception:
        return np.ones(len(ep), dtype=bool)
    keep = np.ones(len(ep), dtype=bool)
    if bad_idx:
        bad_idx = set(int(i) for i in bad_idx)
        keep = np.array([i not in bad_idx for i in range(len(ep))])
    return keep


`_harmonized_filtered_runs` and `_cho2017_qc_mask` above are shared by both export paths defined in Section 5 (the ordinary path, and the RAM-conscious session-by-session path used for Lee2019).

## Section 5 — RAM-managed export loop, all six datasets, all subjects

**RAM policy, and why it is set this way:**

- **RAM-aware parallel by subject (`joblib.Parallel`, `backend="loky"`), not
  sequential.** The main notebook's memory-gate concern doesn't apply the same way
  here: THAT parallelism forked workers on top of a long-lived kernel session that
  had already accumulated large in-memory objects across many earlier cells (raw
  cohorts, figures, dataframes), so every worker inherited that footprint before
  touching its own subject. This exporter's workers do none of that -- each one
  loads its own subject's raw data fresh inside its own process and holds nothing
  else, so peak RAM scales as `n_jobs x per-subject footprint`, not
  `n_jobs x (per-subject + accumulated-session footprint)`. `n_jobs` is picked
  automatically from *live* free RAM using two fields that were already part of
  `RamConfig` but previously unused -- `free_ram_reserve_gb` (floor never crossed)
  and `per_worker_gb_budget` (assumed peak per subject) -- recomputed before every
  chunk, so it adapts if free RAM changes mid-run. Set `ram.n_jobs` to a fixed
  positive integer to override the automatic estimate.
  - **Chunked, not one giant parallel call.** Subjects are dispatched in waves of
    `~3 x n_jobs` so the checkpoint is written after every wave (bounded redo on a
    disconnect, not a full restart) and RAM is re-measured between waves rather
    than only once at the start of a multi-hour job.
  - **Only the main process touches the checkpoint file.** Workers return their
    result; the driving loop is the sole writer, avoiding a multi-process race on
    the same JSON file that letting each worker write to it directly would cause.
  - **Worker pool recycled periodically.** `loky` reuses worker processes across
    waves for efficiency, which is usually fine but can let memory creep from
    allocator fragmentation over hundreds of tasks; the pool is shut down and
    restarted every 5 chunks so that creep cannot accumulate across the whole run.
  - **`threadpool_limits(limits=1)` still wraps each subject's processing**, now
    for a different reason than before: with several subjects running in parallel
    OS processes, letting each one's BLAS calls also spawn multiple threads would
    oversubscribe the machine (`n_jobs x threads-per-subject` instead of `n_jobs`).
- **`del` + `gc.collect()` after every subject.** Each subject holds: raw EDF/MAT
  buffers, filtered copies, the concatenated+CAR'd raw, and the epochs array --
  several times the size of the final stored trial. All of it is discarded before
  the next subject starts.
- **Lee2019 sessions processed separately when `lee2019_process_sessions_separately`
  is on.** Lee2019 is natively 1000 Hz with 62 channels and both offline sessions are
  required; loading both sessions' raw EDF/MAT at once is the single largest memory
  spike in the whole cohort. Loading, filtering and resampling session 1 to
  `target_sfreq`, discarding the 1000 Hz buffer, then doing the same for session 2,
  and only then concatenating the two (already small, already-resampled) results
  keeps peak memory close to one session's worth instead of two. The two paths
  (ordinary and split-by-session) share one tail (`_export_from_processed_runs`)
  so the actual export logic is written once, not duplicated.
- **Resumable (`ram.resume=True`).** Every subject's shard is checked against a
  checkpoint file before doing any work; a disconnect partway through the
  ~300-subject cohort does not require starting over.
- **Failures are logged, not fatal.** One subject's download or processing error is
  recorded in `datasets_summary` and the loop continues.

In [ ]:
def _free_ram_gb() -> float:
    return psutil.virtual_memory().available / (1024 ** 3)


def _load_checkpoint(ckpt_path: Path) -> dict:
    if ckpt_path.exists():
        return json.loads(ckpt_path.read_text())
    return {}


def _save_checkpoint(ckpt_path: Path, state: dict) -> None:
    _write_with_retry(ckpt_path.write_text, json.dumps(state, indent=2, default=str))


def _resolve_subject_ids(cfg, select: "DatasetSelectConfig") -> list:
    '''Subject IDs are 1-indexed (cfg.n_subjects counts them, not a max ID) -- reuse
    the package's OWN resolution helper (pipeline_builder._resolve_subject_ids)
    rather than re-deriving the same range/exclusion logic here.'''
    spec = None if select.subjects == "all" else select.subjects
    usable, _skipped_known_bad = pkgpb._resolve_subject_ids(spec, cfg)
    return usable


def _export_from_processed_runs(cfg, subject, processed_runs, session_run, common_channels,
                                  car_scope_channels, preproc, out_dir, global_subject_idx):
    '''Shared tail for both export paths below: concatenate -> CAR -> epoch -> crop ->
    save. Takes already-harmonized/filtered/resampled runs plus their (session, run)
    provenance tags, so the caller decides HOW those runs were produced (all at once,
    or session-by-session for RAM reasons) without this logic being duplicated.'''
    t0 = time.time()
    boundaries = [0]
    for pr in processed_runs:
        boundaries.append(boundaries[-1] + pr.n_times)

    mask = pkgsp.get_channel_mask(processed_runs[0])
    raw_concat = mne.concatenate_raws([p.copy() for p in processed_runs], preload=True) \
        if len(processed_runs) > 1 else processed_runs[0]
    if mask is not None:
        pkgsp._attach_mask(raw_concat, mask)
    raw_car = pkgsp.set_common_average_reference(raw_concat, car_scope_channels)

    events, _ = mne.events_from_annotations(raw_car, event_id=cfg.event_id, verbose=False)
    if len(events) == 0:
        raise RuntimeError(f"{cfg.name}/S{subject:03d}: no events matched event_id={cfg.event_id}.")

    tmin = preproc.master_window[0] + cfg.cue_offset_sec
    tmax = preproc.master_window[1] + cfg.cue_offset_sec
    ep = pkgsp.make_epochs(raw_car, events=events, event_id=cfg.event_id, tmin=tmin, tmax=tmax, baseline=None)
    if len(ep) == 0:
        raise RuntimeError(f"{cfg.name}/S{subject:03d}: 0 epochs survived cropping.")

    ev_samples = ep.events[:, 0] - raw_car.first_samp
    orig_ev_samples = ep.events[:, 0].copy()
    run_index = np.clip(np.searchsorted(boundaries, ev_samples, side="right") - 1, 0, len(processed_runs) - 1)
    trial_index_in_run = np.zeros(len(ep), dtype=int)
    for ridx in np.unique(run_index):
        sel = np.where(run_index == ridx)[0]
        trial_index_in_run[sel] = np.arange(len(sel))
    run_labels_per_trial = np.array([session_run[i][1] for i in run_index], dtype=object)
    session_per_trial = np.array([session_run[i][0] for i in run_index], dtype=object)

    keep = _cho2017_qc_mask(cfg, subject, ep, ev_samples, run_labels_per_trial)
    if not keep.all():
        idx = np.where(keep)[0]
        ep = ep[idx]
        ev_samples, orig_ev_samples = ev_samples[idx], orig_ev_samples[idx]
        run_index, trial_index_in_run = run_index[idx], trial_index_in_run[idx]
        run_labels_per_trial, session_per_trial = run_labels_per_trial[idx], session_per_trial[idx]

    # MNE's tmax is INCLUSIVE -> one extra trailing sample; truncate to N_SAMPLES explicitly
    # (same convention the package's own PRIMARY_EPOCH_WINDOW uses for its primary window).
    X_volts = ep.get_data(picks=common_channels)[:, :, :N_SAMPLES]
    assert X_volts.shape[-1] == N_SAMPLES, (
        f"{cfg.name}/S{subject:03d}: got {X_volts.shape[-1]} samples, expected {N_SAMPLES}.")
    X = (X_volts * 1e6).astype(np.float32)

    event_codes = ep.events[:, 2]
    label_str = np.array([cfg.label_map[c] for c in event_codes], dtype=object)  # keyed by event CODE
    label_int = np.array([LABEL_MAP[s] for s in label_str], dtype=np.int8)

    ea_reference = None
    if preproc.ea_enabled:
        try:
            ea_reference = pkgalign.compute_ea_reference(X_volts).astype(np.float32)
        except Exception as exc:
            print(f"  [EA] {cfg.name}/S{subject:03d}: reference not computed ({exc}); continuing without it.")

    n_real_channels = int(mask.sum()) if mask is not None else X.shape[1]

    shard_path = out_dir / f"S{subject:03d}.npz"
    shard_payload = dict(
        X=X, y=label_int, run_index=run_index.astype(np.int16),
        trial_index_in_run=trial_index_in_run.astype(np.int16),
        global_trial_index=np.arange(len(ep)).astype(np.int32),
        event_sample_master=ev_samples.astype(np.int32), orig_event_sample=orig_ev_samples.astype(np.int64),
        session=session_per_trial.astype(str), run_label=run_labels_per_trial.astype(str),
        channel_mask=(mask.astype(bool) if mask is not None else np.ones(X.shape[1], bool)),
        channel_names=np.array(common_channels, dtype=object),
    )
    if ea_reference is not None:
        shard_payload["ea_reference"] = ea_reference
    # _write_with_retry is provided by Section 0 (environment setup); on local disk
    # (Kaggle) it is a thin pass-through, on Drive-mounted storage (Colab) it retries
    # transient FUSE I/O errors with backoff -- see that section for why.
    _write_with_retry(np.savez_compressed, shard_path, **shard_payload)
    shard_sha1 = hashlib.sha1(shard_path.read_bytes()).hexdigest()

    trials_rows = [dict(
        trial_uid=f"{cfg.name}_S{subject:03d}_{session_per_trial[i]}_{run_labels_per_trial[i]}_{trial_index_in_run[i]}_{i}",
        dataset=cfg.name, dataset_idx=DATASET_IDX[cfg.name], subject_id=subject,
        global_subject_idx=global_subject_idx, session=session_per_trial[i], run_label=run_labels_per_trial[i],
        run_index=int(run_index[i]), trial_index_in_run=int(trial_index_in_run[i]), global_trial_index=int(i),
        label_str=label_str[i], label_int=int(label_int[i]), event_sample_master=int(ev_samples[i]),
        orig_event_sample=int(orig_ev_samples[i]), n_real_channels=n_real_channels, native_sfreq=cfg.native_sfreq,
        cue_offset_sec=cfg.cue_offset_sec, feedback_onset_sec=cfg.feedback_onset_sec,
        pre_cue_clean_sec=cfg.pre_cue_clean_sec, preproc_id=CFG.preproc.preproc_id(),
        shard_path=str(shard_path.relative_to(PROCESSED_ROOT)), row_in_shard=i,
    ) for i in range(len(ep))]

    n_left, n_right = int((label_str == "left_hand").sum()), int((label_str == "right_hand").sum())
    subject_row = dict(
        dataset=cfg.name, subject_id=subject, global_subject_idx=global_subject_idx,
        n_trials=len(ep), n_trials_left=n_left, n_trials_right=n_right, n_runs=len(processed_runs),
        run_labels=json.dumps(sorted(set(run_labels_per_trial.tolist()))), n_real_channels=n_real_channels,
        native_sfreq=cfg.native_sfreq, shard_path=str(shard_path.relative_to(PROCESSED_ROOT)),
        shard_sha1=shard_sha1, shard_shape=json.dumps(list(X.shape)),
        ea_reference_stored=ea_reference is not None, processing_seconds=round(time.time() - t0, 2),
        export_timestamp=pd.Timestamp.utcnow().isoformat(), mne_version=mne.__version__, moabb_version=moabb.__version__,
    )
    del raw_concat, raw_car, ep, X_volts
    return dict(trials=trials_rows, subject=subject_row)


def export_one_subject(cfg, subject, channel_mapping_df, common_channels, montage,
                        car_scope_channels, preproc, out_dir, global_subject_idx):
    '''Ordinary path: load every run for this subject, harmonize/filter/resample all
    of them, then hand off to the shared tail.'''
    raws = cfg.loader_fn(subject)
    session_run = [_parse_provenance(r) for r in raws]
    processed_runs = _harmonized_filtered_runs(
        raws, channel_mapping_df, common_channels, montage, preproc.bandpass,
        preproc.target_sfreq, preproc.harmonization_mode, preproc.fir_trans_bandwidth)
    del raws
    result = _export_from_processed_runs(cfg, subject, processed_runs, session_run, common_channels,
                                          car_scope_channels, preproc, out_dir, global_subject_idx)
    del processed_runs
    return result


def export_lee2019_subject_split_sessions(cfg, subject, sessions, channel_mapping_df,
                                           common_channels, montage, car_scope_channels,
                                           preproc, out_dir, global_subject_idx):
    '''RAM-conscious path: load, harmonize/filter/resample ONE session at a time,
    discard its 1000 Hz buffer, then do the next session. Only the small, already
    128 Hz results are held together, and only when handing off to the shared tail.'''
    processed_runs, session_run = [], []
    for s in sessions:
        loader = make_lee2019_loader_with_provenance((s,))
        raws = loader(subject)
        session_run.extend(_parse_provenance(r) for r in raws)
        processed_runs.extend(_harmonized_filtered_runs(
            raws, channel_mapping_df, common_channels, montage, preproc.bandpass,
            preproc.target_sfreq, preproc.harmonization_mode, preproc.fir_trans_bandwidth))
        del raws
        gc.collect()
    result = _export_from_processed_runs(cfg, subject, processed_runs, session_run, common_channels,
                                          car_scope_channels, preproc, out_dir, global_subject_idx)
    del processed_runs
    return result


def _safe_n_jobs() -> int:
    """RAM-aware worker count for the parallel export loop below.

    `CFG.ram.free_ram_reserve_gb` and `CFG.ram.per_worker_gb_budget` were already
    fields on `RamConfig` (and already printed in Section 1's config table) but
    were never actually read anywhere -- this is the first place that uses them.
    An explicit `CFG.ram.n_jobs` (any value other than the default `1`) always
    wins over the estimate, so this is opt-in unless left at its default.
    """
    if CFG.ram.n_jobs != 1:
        return max(1, int(CFG.ram.n_jobs))
    free_gb = _free_ram_gb()
    usable_gb = max(free_gb - CFG.ram.free_ram_reserve_gb, 0.0)
    n = int(usable_gb / max(CFG.ram.per_worker_gb_budget, 0.1))
    n = max(1, min(n, os.cpu_count() or 1, 8))   # never exceed cores, or an 8-worker ceiling
    return n


def _export_one_task(cfg, sid, channel_mapping_df, common_channels, montage, car_scope_channels,
                      preproc, out_dir, global_subject_idx, lee2019_sessions, split_lee2019_sessions):
    """Runs inside a `loky` worker process. Returns (key, result_dict) instead of
    touching any shared state -- the driving process is the ONLY thing that writes
    to `checkpoint`/`all_subject_rows`/`all_trials_rows`, so concurrent workers
    never race on them. Errors are caught here (not left to propagate through
    `joblib`) so one subject's failure cannot abort the whole chunk."""
    key = f"{cfg.name}:{sid}"
    try:
        with threadpool_limits(limits=1):
            if cfg.name == "lee2019" and split_lee2019_sessions:
                out = export_lee2019_subject_split_sessions(
                    cfg, sid, lee2019_sessions, channel_mapping_df, common_channels,
                    montage, car_scope_channels, preproc, out_dir, global_subject_idx)
            else:
                out = export_one_subject(cfg, sid, channel_mapping_df, common_channels,
                                          montage, car_scope_channels, preproc, out_dir,
                                          global_subject_idx)
        gc.collect()
        return key, dict(status="ok", subject=out["subject"], trials=out["trials"])
    except Exception as exc:
        gc.collect()
        return key, dict(status="failed", reason=str(exc))


In [ ]:
# --- Resolve which subjects will actually be exported, and assign global_subject_idx --
subject_plan = {}   # dataset -> list[subject_id]
for cfg in registry.enabled():
    subject_plan[cfg.name] = _resolve_subject_ids(cfg, CFG.datasets[cfg.name])

next_idx = 0
for dname in sorted(subject_plan):
    for sid in subject_plan[dname]:
        GLOBAL_SUBJECT_IDX[(dname, sid)] = next_idx
        next_idx += 1

total_planned = sum(len(v) for v in subject_plan.values())
print("Subjects planned for export:")
for dname, sids in subject_plan.items():
    print(f"  {dname:13s} {len(sids):4d} subjects")
print(f"  TOTAL          {total_planned:4d} subjects  (global_subject_idx 0..{next_idx-1})")


Subjects planned for export:
  physionet      105 subjects
  cho2017         52 subjects
  lee2019         54 subjects
  bnci2014001      9 subjects
  weibo2014       10 subjects
  dreyer2023      86 subjects
  TOTAL           316 subjects  (global_subject_idx 0..315)


In [ ]:
from joblib import Parallel, delayed

CKPT_PATH = PROCESSED_ROOT / "_export_checkpoint.json"
checkpoint = _load_checkpoint(CKPT_PATH) if CFG.ram.resume else {}

all_trials_rows, all_subject_rows = [], []
dataset_summary = {d: {"n_registry": 0, "n_excluded_by_config": 0, "n_exported": 0,
                        "n_failed": 0, "failed": {}} for d in subject_plan}

# --- Build the full remaining-work task list up front, exactly the same
#     resume/skip logic the sequential loop used to apply per-subject. ---
tasks = []   # (cfg, sid, out_dir)
for cfg in registry.enabled():
    dname = cfg.name
    out_dir = DATA_ROOT / dname
    out_dir.mkdir(parents=True, exist_ok=True)
    sids = subject_plan[dname]
    dataset_summary[dname]["n_registry"] = cfg.n_subjects
    dataset_summary[dname]["n_excluded_by_config"] = cfg.n_subjects - len(sids)
    for sid in sids:
        key = f"{dname}:{sid}"
        expected_shard = out_dir / f"S{sid:03d}.npz"
        if CFG.ram.resume and checkpoint.get(key, {}).get("status") == "ok" and expected_shard.exists():
            row = checkpoint[key]
            all_subject_rows.append(row["subject"])
            all_trials_rows.extend(row["trials"])
            continue
        tasks.append((cfg, sid, out_dir))

n_jobs = _safe_n_jobs()
n_jobs = 2
print(f"Parallel export: {len(tasks)} subject(s) remaining (of {total_planned} planned), "
      f"n_jobs={n_jobs} (free RAM {_free_ram_gb():.1f} GB, "
      f"reserve {CFG.ram.free_ram_reserve_gb} GB, budget/worker {CFG.ram.per_worker_gb_budget} GB).")

CHUNK = max(n_jobs, 1) * 2   # a few waves per worker: frequent enough checkpoint
                             # writes to bound redo-on-disconnect, without losing
                             # most of the parallel speedup to per-chunk overhead
for chunk_i, chunk_start in enumerate(range(0, len(tasks), CHUNK)):
    chunk = tasks[chunk_start: chunk_start + CHUNK]

    if _free_ram_gb() < CFG.ram.free_ram_reserve_gb:
        gc.collect()
        print(f"  [RAM] {_free_ram_gb():.2f} GB free, below reserve; forced gc.collect() "
              f"before chunk {chunk_i + 1}.")
        n_jobs = _safe_n_jobs()   # re-check -- RAM pressure can ease or worsen between chunks

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_export_one_task)(cfg, sid, channel_mapping_df, common_channels, montage,
                                  car_scope, CFG.preproc, out_dir,
                                  GLOBAL_SUBJECT_IDX[(cfg.name, sid)],
                                  CFG.datasets["lee2019"].lee2019_sessions,
                                  CFG.ram.lee2019_process_sessions_separately)
        for cfg, sid, out_dir in chunk
    )

    for (cfg, sid, out_dir), (key, res) in zip(chunk, results):
        dname = cfg.name
        checkpoint[key] = res   # main process only -- no concurrent writers
        if res["status"] == "ok":
            all_subject_rows.append(res["subject"])
            all_trials_rows.extend(res["trials"])
            dataset_summary[dname]["n_exported"] += 1
            print(f"  [{dname:11s} S{sid:03d}] {res['subject']['n_trials']:3d} trials, "
                  f"{res['subject']['n_runs']} runs, {res['subject']['processing_seconds']:.1f}s")
        else:
            dataset_summary[dname]["n_failed"] += 1
            dataset_summary[dname]["failed"][sid] = res["reason"]
            print(f"  [{dname:11s} S{sid:03d}] FAILED: {res['reason']}")

    if CFG.ram.gc_every_subject:
        gc.collect()
    _save_checkpoint(CKPT_PATH, checkpoint)
    print(f"  [chunk {chunk_i + 1}] done, {_free_ram_gb():.1f} GB free.")

    # Recycle the loky worker pool periodically so memory cannot slowly creep
    # (allocator fragmentation across many reused worker processes) over a
    # multi-hour, ~300-subject run.
    if chunk_i % 5 == 4:
        from joblib.externals.loky import get_reusable_executor
        get_reusable_executor().shutdown(wait=True)

print("\nExport loop complete.")
print(f"Exported subjects: {len(all_subject_rows)} / {total_planned}")

Parallel export: 94 subject(s) remaining (of 316 planned), n_jobs=4 (free RAM 10.4 GB, reserve 2.0 GB, budget/worker 1.5 GB).
  [weibo2014   S003] 160 trials, 1 runs, 5.9s
  [weibo2014   S004] 160 trials, 1 runs, 6.3s
  [weibo2014   S005] 160 trials, 1 runs, 7.5s
  [weibo2014   S006] 140 trials, 1 runs, 7.4s
  [weibo2014   S007] 160 trials, 1 runs, 8.4s
  [weibo2014   S008] 160 trials, 1 runs, 7.4s
  [weibo2014   S009] 160 trials, 1 runs, 6.1s
  [weibo2014   S010] 160 trials, 1 runs, 7.4s
  [dreyer2023  S001] 240 trials, 6 runs, 2.8s
  [dreyer2023  S002] 240 trials, 6 runs, 5.1s
  [dreyer2023  S003] 240 trials, 6 runs, 4.2s
  [dreyer2023  S004] 240 trials, 6 runs, 3.0s
  [chunk 1] done, 9.1 GB free.
  [dreyer2023  S005] 240 trials, 6 runs, 4.4s
  [dreyer2023  S006] 240 trials, 6 runs, 4.8s
  [dreyer2023  S007] 240 trials, 6 runs, 4.3s
  [dreyer2023  S008] 240 trials, 6 runs, 4.1s
  [dreyer2023  S009] 240 trials, 6 runs, 4.6s
  [dreyer2023  S010] 240 trials, 6 runs, 4.9s
  [dreyer2023  

## Section 6 — Consolidate metadata tables

In [ ]:
trials_df = pd.DataFrame(all_trials_rows)
subjects_df = pd.DataFrame(all_subject_rows)
datasets_summary_df = pd.DataFrame([
    dict(dataset=d, **{k: v for k, v in s.items() if k != "failed"},
         failed_subjects=json.dumps(s["failed"]))
    for d, s in dataset_summary.items()
])

_write_with_retry(trials_df.to_parquet, META_ROOT / "trials.parquet", index=False)
_write_with_retry(subjects_df.to_parquet, META_ROOT / "subjects.parquet", index=False)
_write_with_retry(datasets_summary_df.to_csv, META_ROOT / "datasets_summary.csv", index=False)

_write_with_retry(Path(CONFIG_ROOT / "experiment_config.json").write_text,
              json.dumps(CFG.to_jsonable(), indent=2))
_write_with_retry((CONFIG_ROOT / "preproc_id.txt").write_text, CFG.preproc.preproc_id())
_write_with_retry((CHANNELS_ROOT / "target_channels.json").write_text, json.dumps(TARGET_CHANNELS))
_write_with_retry(np.savez, CHANNELS_ROOT / "channel_masks.npz", **{k: v for k, v in channel_masks.items()})
_write_with_retry(channel_mapping_df.to_csv, CHANNELS_ROOT / "channel_mapping_table.csv", index=False)

print(f"trials.parquet:   {len(trials_df):6d} rows")
print(f"subjects.parquet: {len(subjects_df):6d} rows")
display(datasets_summary_df)


trials.parquet:    50847 rows
subjects.parquet:    316 rows


,dataset,n_registry,n_excluded_by_config,n_exported,n_failed,failed_subjects
0,physionet,109,4,0,0,{}
1,cho2017,52,0,0,0,{}
2,lee2019,54,0,0,0,{}
3,bnci2014001,9,0,0,0,{}
4,weibo2014,10,0,8,0,{}
5,dreyer2023,87,1,86,0,{}


## Section 7 — QC suite

Every check below runs against what was actually written to `processed/`, reading
shards back from disk rather than trusting in-memory state from the export loop --
this is deliberately a check on the *artifact*, not a check on the code that produced
it. Every result is collected into one `qc_report` and nothing here halts execution;
`qc_status` is decided once, at the end, from the full set of results.

In [ ]:
qc_results = {"checks": [], "per_subject": [], "status": "pending"}

def _qc_record(name, passed, detail, severity="fail"):
    qc_results["checks"].append(dict(name=name, passed=bool(passed), severity=severity, detail=detail))
    mark = "PASS" if passed else severity.upper()
    print(f"  [{mark:4s}] {name}: {detail}")


In [ ]:
print("QC 1 -- shape, finiteness, mask-zero, amplitude range, class balance (per subject)")
amp_min, amp_max = CFG.qc.amplitude_median_abs_uv_min, CFG.qc.amplitude_median_abs_uv_max

for _, row in subjects_df.iterrows():
    shard = np.load(PROCESSED_ROOT / row["shard_path"], allow_pickle=True)
    X, y, mask = shard["X"], shard["y"], shard["channel_mask"]
    issues = []

    if X.shape != tuple(json.loads(row["shard_shape"])):
        issues.append(f"shape mismatch: on-disk {X.shape} vs metadata {row['shard_shape']}")
    if not np.isfinite(X).all():
        issues.append(f"non-finite values: {np.isnan(X).sum()} nan, {np.isinf(X).sum()} inf")
    if CFG.qc.require_masked_exactly_zero and (~mask).any():
        if not np.allclose(X[:, ~mask, :], 0.0):
            issues.append("masked channels are not exactly zero")
    real_med_abs = float(np.median(np.abs(X[:, mask, :]))) if mask.any() else float("nan")
    if not (amp_min <= real_med_abs <= amp_max):
        issues.append(f"median|X| over real channels = {real_med_abs:.3f} uV, outside [{amp_min},{amp_max}]")
    n0, n1 = int((y == 0).sum()), int((y == 1).sum())
    if min(n0, n1) < CFG.qc.min_trials_per_class:
        issues.append(f"class {['left','right'][int(n0<n1)]} has only {min(n0,n1)} trials")
    if n0 > 0 and n1 > 0 and max(n0, n1) / max(1, min(n0, n1)) > CFG.qc.max_class_imbalance_ratio:
        issues.append(f"class imbalance ratio {max(n0,n1)/max(1,min(n0,n1)):.2f} exceeds threshold")

    qc_results["per_subject"].append(dict(
        dataset=row["dataset"], subject_id=int(row["subject_id"]), median_abs_uv=real_med_abs,
        n_left=n0, n_right=n1, n_real_channels=int(mask.sum()), issues=issues,
        qc_status="fail" if issues else "pass"))

n_subj_fail = sum(1 for r in qc_results["per_subject"] if r["qc_status"] == "fail")
_qc_record("per_subject_checks", n_subj_fail == 0,
           f"{n_subj_fail}/{len(subjects_df)} subject(s) flagged -- see qc_results['per_subject']")


QC 1 -- shape, finiteness, mask-zero, amplitude range, class balance (per subject)
  [PASS] per_subject_checks: 0/316 subject(s) flagged -- see qc_results['per_subject']


In [ ]:
print("\nQC 2 -- per-dataset channel-mask invariant (every subject of a dataset shares one mask)")
for dname in subjects_df["dataset"].unique():
    sub_rows = subjects_df[subjects_df["dataset"] == dname]
    masks = [np.load(PROCESSED_ROOT / r["shard_path"], allow_pickle=True)["channel_mask"]
             for _, r in sub_rows.iterrows()]
    all_equal = all(np.array_equal(masks[0], m) for m in masks)
    _qc_record(f"channel_mask_invariant[{dname}]", all_equal,
               f"{len(masks)} subject(s) checked" if all_equal else "mask differs across subjects")

_mask_lengths_by_dataset = {}
for dname in subjects_df["dataset"].unique():
    sub_rows = subjects_df[subjects_df["dataset"] == dname]
    lens = {len(np.load(PROCESSED_ROOT / r["shard_path"], allow_pickle=True)["channel_mask"])
            for _, r in sub_rows.iterrows()}
    _mask_lengths_by_dataset[dname] = lens

_expected_mask_len = len(TARGET_CHANNELS)
_mismatched_datasets = {d: lens for d, lens in _mask_lengths_by_dataset.items()
                        if lens != {_expected_mask_len}}
_qc_record("channel_mask_length_consistent_across_cohort", not _mismatched_datasets,
           f"expected length {_expected_mask_len} (current TARGET_CHANNELS) for every "
           f"subject in every dataset; " +
           (f"mismatch found in: {_mismatched_datasets}" if _mismatched_datasets
            else f"confirmed for all {len(subjects_df)} subjects"))
if _mismatched_datasets:
    print(f"  -> stale shards likely exist for: {list(_mismatched_datasets)}. Re-export "
          f"those dataset(s) (delete their .npz files first so `ram.resume` cannot skip "
          f"them) under the CURRENT Section 2 configuration before trusting QC 3 or "
          f"training on this export.")



QC 2 -- per-dataset channel-mask invariant (every subject of a dataset shares one mask)
  [PASS] channel_mask_invariant[physionet]: 105 subject(s) checked
  [PASS] channel_mask_invariant[cho2017]: 52 subject(s) checked
  [PASS] channel_mask_invariant[lee2019]: 54 subject(s) checked
  [PASS] channel_mask_invariant[bnci2014001]: 9 subject(s) checked
  [PASS] channel_mask_invariant[weibo2014]: 10 subject(s) checked
  [PASS] channel_mask_invariant[dreyer2023]: 86 subject(s) checked
  [PASS] channel_mask_length_consistent_across_cohort: expected length 62 (current TARGET_CHANNELS) for every subject in every dataset; confirmed for all 316 subjects


In [ ]:
print("\nQC 3 -- dataset-identity shortcut probe (diagnostic, not a failure by itself)")
# Concatenated masks are known to near-perfectly identify dataset
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

mask_features, dataset_labels, _n_skipped_len_mismatch = [], [], 0
for _, row in subjects_df.iterrows():
    shard = np.load(PROCESSED_ROOT / row["shard_path"], allow_pickle=True)
    m = shard["channel_mask"].astype(float)

    if len(m) != len(TARGET_CHANNELS):
        _n_skipped_len_mismatch += 1
        continue
    mask_features.append(m)
    dataset_labels.append(row["dataset"])
if _n_skipped_len_mismatch:
    print(f"  [WARNING] {_n_skipped_len_mismatch} subject(s) skipped for this probe due "
          f"to a channel_mask length mismatch -- see QC 2's "
          f"'channel_mask_length_consistent_across_cohort' result above for which "
          f"dataset(s) need re-exporting.")
mask_features = np.array(mask_features)
if len(set(dataset_labels)) > 1:
    acc = cross_val_score(LogisticRegression(max_iter=1000), mask_features, dataset_labels, cv=5).mean()
    _qc_record("dataset_identity_from_mask", True,
               f"{acc*100:.1f}% dataset-identity accuracy from the channel mask alone "
               f"(expected near 100%; use the 'shared' channel view at load time if this "
               f"shortcut is a concern for your model)", severity="info")



QC 3 -- dataset-identity shortcut probe (diagnostic, not a failure by itself)
  [PASS] dataset_identity_from_mask: 83.5% dataset-identity accuracy from the channel mask alone (expected near 100%; use the 'shared' channel view at load time if this shortcut is a concern for your model)


In [ ]:
print("\nQC 4 -- determinism (re-run a few subjects end-to-end, compare byte-for-byte)")
import random
rng = random.Random(CFG.seed)
check_pool = [(cfg, sid) for cfg in registry.enabled() for sid in subject_plan[cfg.name]
              if dataset_summary[cfg.name]["failed"].get(sid) is None]
sample = rng.sample(check_pool, min(CFG.qc.n_determinism_check_subjects, len(check_pool)))

det_ok = True
for cfg, sid in sample:
    orig_shard = np.load(DATA_ROOT / cfg.name / f"S{sid:03d}.npz", allow_pickle=True)
    with threadpool_limits(limits=1):
        if cfg.name == "lee2019" and CFG.ram.lee2019_process_sessions_separately:
            rerun = export_lee2019_subject_split_sessions(
                cfg, sid, CFG.datasets["lee2019"].lee2019_sessions, channel_mapping_df,
                common_channels, montage, car_scope, CFG.preproc, OUTPUT_ROOT, -1)
        else:
            rerun = export_one_subject(cfg, sid, channel_mapping_df, common_channels,
                                        montage, car_scope, CFG.preproc, OUTPUT_ROOT, -1)
    rerun_shard = np.load(OUTPUT_ROOT / f"S{sid:03d}.npz", allow_pickle=True)
    same = np.allclose(orig_shard["X"], rerun_shard["X"]) and np.array_equal(orig_shard["y"], rerun_shard["y"])
    det_ok &= same
    (OUTPUT_ROOT / f"S{sid:03d}.npz").unlink(missing_ok=True)
    _qc_record(f"determinism[{cfg.name}/S{sid:03d}]", same, "re-export matches stored shard exactly")

qc_results["status"] = "pass" if (n_subj_fail == 0 and det_ok
                                   and all(c["passed"] for c in qc_results["checks"] if c["severity"] == "fail")) else "fail"
_write_with_retry((QC_ROOT / "qc_report.json").write_text, json.dumps(qc_results, indent=2, default=str))
print(f"\nQC overall status: {qc_results['status'].upper()}")


## Section 8 — Leakage-free split manifests and validation

This section builds subject-level split manifests and then **proves** the thing that
matters most for every later experiment: that grouping by subject actually behaves
differently from not grouping. A split manifest can look correct (disjoint subject
lists) while still being useless if the downstream training code ignores it -- the
demonstration below is a second, independent check that operates on the data itself,
using a simple classifier, not on the manifest's bookkeeping.

Three things are checked, each with a pass/fail:

1. **Structural leakage** -- locked-test subjects never appear in any fold; folds
   within a dataset are disjoint; every enabled subject is accounted for exactly once
   (either in a fold or in the locked-test set).
2. **Grouped-vs-ungrouped contrast** -- a simple band-power + logistic-regression
   classifier is scored two ways on the same trials: (a) `StratifiedGroupKFold`
   grouped by subject, and (b) plain trial-level `StratifiedKFold` with no grouping.
   If (b) scores meaningfully higher than (a), trials from the same subject are
   leaking across the ungrouped fold boundary -- exactly the effect subject-grouping
   exists to prevent. This is the leakage-audit contrast the roadmap and Stage-0
   notes describe (grouped CV showing markedly lower, more honest accuracy than
   ungrouped CV on the same data).
3. **Locked-test isolation** -- confirms no code above this cell has touched the
   locked-test subjects' trials for anything other than writing their shard to disk
   (no fold, no probe, no threshold tuned against them).


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from scipy.signal import welch

split_rng = np.random.RandomState(CFG.split.seed)
subjects_df["subject_key"] = subjects_df["dataset"] + ":" + subjects_df["subject_id"].astype(str)

# --- Locked test subjects: reserved once, stratified per dataset, never used below ---
EXPLORATION_TAINTED_SUBJECTS = {
    "physionet":   set(range(1, 11)),
    "cho2017":     set(range(1, 11)),
    "lee2019":     {1, 2},
    "bnci2014001": set(range(1, 7)),
    "weibo2014":   set(range(1, 8)),
    "dreyer2023":  set(range(1, 11)),
}  # subject IDs consumed by the earlier preprocessing-design notebook's ablations
   # (n_subjects_per_dataset = physionet:10, cho2017:10, lee2019:[1,2],
   # bnci2014001:6, weibo2014:7, dreyer2023:10) to choose harmonization_mode,
   # car_scope, bandpass and artifact_variant -- all fixed as constants above,
   # before this split, and before this notebook, existed.

subjects_df["exploration_tainted"] = subjects_df.apply(
    lambda r: int(r["subject_id"]) in EXPLORATION_TAINTED_SUBJECTS.get(r["dataset"], set()), axis=1)

locked_test_keys = []
for dname, grp in subjects_df.groupby("dataset"):
    eligible = grp.loc[~grp["exploration_tainted"], "subject_key"].tolist()
    tainted = grp.loc[grp["exploration_tainted"], "subject_key"].tolist()
    split_rng.shuffle(eligible)
    split_rng.shuffle(tainted)
    n_locked = max(1, int(round(len(grp) * CFG.split.locked_test_frac))) if len(grp) > 1 else 0
    if len(eligible) >= n_locked:
        locked_test_keys.extend(eligible[:n_locked])
    else:
        # Not enough exploration-untainted subjects to fill the quota for this
        # dataset -- fall back to tainted ones rather than silently under-filling,
        # but say so loudly every time it happens rather than staying quiet.
        shortfall = n_locked - len(eligible)
        print(f"  [WARNING] {dname}: only {len(eligible)} exploration-untainted "
              f"subject(s) available, need {n_locked} for locked_test_frac="
              f"{CFG.split.locked_test_frac}. Falling back to {shortfall} "
              f"exploration-tainted subject(s) for this dataset only.")
        locked_test_keys.extend(eligible)
        locked_test_keys.extend(tainted[:shortfall])
locked_test_keys = set(locked_test_keys)

cv_pool_df = subjects_df[~subjects_df["subject_key"].isin(locked_test_keys)].reset_index(drop=True)
n_tainted_in_locked = subjects_df.loc[subjects_df["subject_key"].isin(locked_test_keys),
                                      "exploration_tainted"].sum()
print(f"Locked test subjects: {len(locked_test_keys)} / {len(subjects_df)} "
      f"({len(cv_pool_df)} remain in the CV pool); "
      f"{int(n_tainted_in_locked)} of them are exploration-tainted "
      f"(0 unless a [WARNING] fallback fired above).")


In [ ]:
# --- Pooled, subject-grouped, stratified-by-dataset K folds over the CV pool ---------
skf_groups = cv_pool_df["subject_key"].values
skf_strata = cv_pool_df["dataset"].values if CFG.split.stratify_by_dataset else np.zeros(len(cv_pool_df))

sgkf = StratifiedGroupKFold(n_splits=CFG.split.n_folds, shuffle=True, random_state=CFG.split.seed)
pooled_folds = []
for fold_i, (train_idx, test_idx) in enumerate(sgkf.split(cv_pool_df, skf_strata, groups=skf_groups)):
    train_keys = set(cv_pool_df.iloc[train_idx]["subject_key"])
    val_keys = set(split_rng.choice(sorted(train_keys),
                    size=max(1, int(round(len(train_keys) * CFG.split.val_frac))), replace=False))
    train_keys -= val_keys
    test_keys = set(cv_pool_df.iloc[test_idx]["subject_key"])
    pooled_folds.append(dict(fold=fold_i, train=sorted(train_keys), val=sorted(val_keys), test=sorted(test_keys)))

within_dataset_folds = {}
for dname, grp in cv_pool_df.groupby("dataset"):
    if len(grp) < CFG.split.n_folds:
        print(f"  [SKIP] within-dataset folds for '{dname}': only {len(grp)} subjects, "
              f"fewer than n_folds={CFG.split.n_folds}.")
        continue
    keys = grp["subject_key"].values

    class_ratio = (grp["n_trials_left"] / grp["n_trials"].replace(0, np.nan)).fillna(0.5)
    if class_ratio.nunique() >= 2:
        kf_labels = pd.qcut(class_ratio, q=2, labels=False, duplicates="drop").values
    else:
        kf_labels = np.zeros(len(grp), dtype=int)
    inner = StratifiedGroupKFold(n_splits=CFG.split.n_folds, shuffle=True, random_state=CFG.split.seed)
    folds = []
    for fold_i, (train_idx, test_idx) in enumerate(inner.split(grp, kf_labels, groups=keys)):
        train_keys = set(keys[train_idx])
        val_keys = set(split_rng.choice(sorted(train_keys),
                        size=max(1, int(round(len(train_keys) * CFG.split.val_frac))), replace=False))
        train_keys -= val_keys
        folds.append(dict(fold=fold_i, train=sorted(train_keys), val=sorted(val_keys),
                           test=sorted(set(keys[test_idx]))))
    within_dataset_folds[dname] = folds

split_manifest = dict(
    protocol=CFG.split.protocol, seed=CFG.split.seed, preproc_id=CFG.preproc.preproc_id(),
    locked_test=sorted(locked_test_keys), pooled_group_kfold=pooled_folds,
    within_dataset=within_dataset_folds)
_write_with_retry((SPLITS_ROOT / "pooled_group_kfold.json").write_text, json.dumps(split_manifest, indent=2))
for dname, folds in within_dataset_folds.items():
    (SPLITS_ROOT / "within_dataset").mkdir(exist_ok=True)
    _write_with_retry((SPLITS_ROOT / "within_dataset" / f"{dname}.json").write_text, json.dumps(folds, indent=2))
_write_with_retry((SPLITS_ROOT / "locked_test_subjects.json").write_text, json.dumps(sorted(locked_test_keys), indent=2))

print(f"Pooled folds: {len(pooled_folds)}   Within-dataset protocols: {list(within_dataset_folds)}")


In [33]:
print("STRUCTURAL LEAKAGE CHECKS")
validation_results = {"checks": [], "status": "pending"}

def _v_record(name, passed, detail):
    validation_results["checks"].append(dict(name=name, passed=bool(passed), detail=detail))
    print(f"  [{'PASS' if passed else 'FAIL':4s}] {name}: {detail}")

for fold in pooled_folds:
    tr, va, te = set(fold["train"]), set(fold["val"]), set(fold["test"])
    _v_record(f"pooled_fold_{fold['fold']}_disjoint", not (tr & va) and not (tr & te) and not (va & te),
               "train/val/test pairwise disjoint")
    _v_record(f"pooled_fold_{fold['fold']}_no_locked_test", not ((tr | va | te) & locked_test_keys),
               "locked-test subjects absent from this fold")

all_cv_keys = set(cv_pool_df["subject_key"])
covered = set()
for fold in pooled_folds:
    covered |= set(fold["train"]) | set(fold["val"]) | set(fold["test"])
_v_record("pooled_full_coverage", covered == all_cv_keys,
           f"{len(covered)}/{len(all_cv_keys)} CV-pool subjects appear in at least one fold")

_v_record("locked_test_disjoint_from_all_subjects", locked_test_keys.issubset(set(subjects_df["subject_key"])),
           f"{len(locked_test_keys)} locked-test subjects, all present in the exported cohort")


STRUCTURAL LEAKAGE CHECKS
  [PASS] pooled_fold_0_disjoint: train/val/test pairwise disjoint
  [PASS] pooled_fold_0_no_locked_test: locked-test subjects absent from this fold
  [PASS] pooled_fold_1_disjoint: train/val/test pairwise disjoint
  [PASS] pooled_fold_1_no_locked_test: locked-test subjects absent from this fold
  [PASS] pooled_fold_2_disjoint: train/val/test pairwise disjoint
  [PASS] pooled_fold_2_no_locked_test: locked-test subjects absent from this fold
  [PASS] pooled_fold_3_disjoint: train/val/test pairwise disjoint
  [PASS] pooled_fold_3_no_locked_test: locked-test subjects absent from this fold
  [PASS] pooled_fold_4_disjoint: train/val/test pairwise disjoint
  [PASS] pooled_fold_4_no_locked_test: locked-test subjects absent from this fold
  [PASS] pooled_full_coverage: 268/268 CV-pool subjects appear in at least one fold
  [PASS] locked_test_disjoint_from_all_subjects: 48 locked-test subjects, all present in the exported cohort


In [34]:
print("\nGROUPED vs UNGROUPED LEAKAGE CONTRAST (the decisive check)")
# A small, fast, dependency-light classifier: log band-power (mu 8-13 Hz, beta 13-30 Hz)
# per channel, LogisticRegression. This is not a model candidate -- it exists only to
# make the leakage effect visible with something quick to fit many times over.
def _bandpower_features(X, sfreq):
    freqs, psd = welch(X, fs=sfreq, axis=-1, nperseg=min(128, X.shape[-1]))
    mu = psd[:, :, (freqs >= 8) & (freqs <= 13)].mean(axis=-1)
    beta = psd[:, :, (freqs >= 13) & (freqs <= 30)].mean(axis=-1)
    return np.log(np.concatenate([mu, beta], axis=1) + 1e-12)

leak_rows = []
for dname, grp in cv_pool_df.groupby("dataset"):
    if grp["subject_id"].nunique() < CFG.split.n_folds or len(grp) < 8:
        continue
    Xs, ys, groups = [], [], []
    for _, row in grp.iterrows():
        shard = np.load(PROCESSED_ROOT / row["shard_path"], allow_pickle=True)
        mask = shard["channel_mask"]
        feats = _bandpower_features(shard["X"][:, mask, :], CFG.preproc.target_sfreq)
        Xs.append(feats); ys.append(shard["y"]); groups.extend([row["subject_key"]] * len(shard["y"]))
    Xs, ys, groups = np.concatenate(Xs), np.concatenate(ys), np.array(groups)
    if len(set(ys)) < 2:
        continue

    grouped_cv = StratifiedGroupKFold(n_splits=min(5, grp["subject_id"].nunique()), shuffle=True, random_state=CFG.split.seed)
    grouped_acc = cross_val_score(LogisticRegression(max_iter=1000), Xs, ys,
                                   groups=groups, cv=grouped_cv).mean()
    ungrouped_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.split.seed)
    ungrouped_acc = cross_val_score(LogisticRegression(max_iter=1000), Xs, ys, cv=ungrouped_cv).mean()

    leak_rows.append(dict(dataset=dname, n_subjects=grp["subject_id"].nunique(), n_trials=len(ys),
                           grouped_acc=grouped_acc, ungrouped_acc=ungrouped_acc,
                           gap=ungrouped_acc - grouped_acc))
    print(f"  {dname:13s} grouped={grouped_acc*100:5.1f}%  ungrouped={ungrouped_acc*100:5.1f}%  "
          f"gap={100*(ungrouped_acc-grouped_acc):+5.1f} pts")

leak_df = pd.DataFrame(leak_rows)
display(leak_df)

# A positive, non-trivial gap is EXPECTED and desired here -- it is the proof that
# subject grouping matters for this data. Its absence (gap ~ 0) would mean subject
# identity carries no decodable signal for this simple probe, which is itself worth
# knowing but does not on its own indicate a leakage problem in the SPLIT MANIFEST.
# What this check guards against is the manifest silently failing to group at all.
manifest_uses_grouping = all(
    len(set(f["train"]) & set(f["test"])) == 0 for f in pooled_folds)
_v_record("split_manifest_enforces_subject_grouping", manifest_uses_grouping,
           "no subject appears in both train and test of any pooled fold")
_v_record("grouped_vs_ungrouped_measured", len(leak_df) > 0,
           f"measured on {len(leak_df)} dataset(s): " +
           (f"mean gap {leak_df['gap'].mean()*100:+.1f} pts" if len(leak_df) else "insufficient subjects to measure"))



GROUPED vs UNGROUPED LEAKAGE CONTRAST (the decisive check)
  bnci2014001   grouped= 70.7%  ungrouped= 77.2%  gap= +6.5 pts
  cho2017       grouped= 59.4%  ungrouped= 63.6%  gap= +4.2 pts
  dreyer2023    grouped= 65.5%  ungrouped= 66.0%  gap= +0.5 pts
  lee2019       grouped= 66.7%  ungrouped= 69.7%  gap= +2.9 pts
  physionet     grouped= 62.8%  ungrouped= 65.0%  gap= +2.2 pts
  weibo2014     grouped= 62.5%  ungrouped= 73.6%  gap=+11.1 pts


,dataset,n_subjects,n_trials,grouped_acc,ungrouped_acc,gap
0,bnci2014001,8,2304,0.707292,0.772139,0.064847
1,cho2017,44,8920,0.593663,0.635538,0.041875
2,dreyer2023,73,17520,0.655107,0.660046,0.004939
3,lee2019,46,9200,0.667389,0.696739,0.029350
4,physionet,89,4003,0.627841,0.650253,0.022412
5,weibo2014,8,1260,0.624542,0.735714,0.111173


  [PASS] split_manifest_enforces_subject_grouping: no subject appears in both train and test of any pooled fold
  [PASS] grouped_vs_ungrouped_measured: measured on 6 dataset(s): mean gap +4.6 pts


### Gate — write final status

In [ ]:
validation_results["status"] = "pass" if all(c["passed"] for c in validation_results["checks"]) else "fail"
_write_with_retry((VALIDATION_ROOT / "leakage_validation_report.json").write_text,
              json.dumps(validation_results, indent=2, default=str))
_write_with_retry((VALIDATION_ROOT / "grouped_vs_ungrouped.csv").write_text,
              leak_df.to_csv(index=False) if len(leak_df) else "")

qc_pass = qc_results["status"] == "pass"
val_pass = validation_results["status"] == "pass"
export_complete = len(all_subject_rows) == total_planned

manifest = dict(
    run_name=CFG.run_name, preproc_id=CFG.preproc.preproc_id(),
    export_timestamp=pd.Timestamp.utcnow().isoformat(),
    n_datasets=len(subject_plan), n_subjects_planned=total_planned, n_subjects_exported=len(all_subject_rows),
    n_trials=len(trials_df), qc_status=qc_results["status"], validation_status=validation_results["status"],
    dataset_summary={d: {k: v for k, v in s.items() if k != "failed"} for d, s in dataset_summary.items()},
)
_write_with_retry((PROCESSED_ROOT / "manifest.json").write_text, json.dumps(manifest, indent=2, default=str))

if qc_pass and val_pass and export_complete:
    _write_with_retry((PROCESSED_ROOT / "STATUS.txt").write_text, "READY_FOR_TRAINING\n")
    print("=" * 70)
    print(f"STATUS: READY_FOR_TRAINING")
    print(f"  {len(all_subject_rows)} subjects, {len(trials_df)} trials, "
          f"preproc_id={CFG.preproc.preproc_id()}")
    print(f"  processed/ -> {PROCESSED_ROOT}")
    print("=" * 70)
else:
    reasons = []
    if not export_complete:
        reasons.append(f"export incomplete: {len(all_subject_rows)}/{total_planned} subjects "
                        f"({sum(s['n_failed'] for s in dataset_summary.values())} failed)")
    if not qc_pass:
        reasons.append("QC gate failed -- see qc/qc_report.json")
    if not val_pass:
        reasons.append("leakage validation failed -- see validation/leakage_validation_report.json")
    _write_with_retry((PROCESSED_ROOT / "STATUS.txt").write_text, "FAILED\n" + "\n".join(reasons) + "\n")
    print("=" * 70)
    print("STATUS: FAILED -- do not train on this export.")
    for r in reasons:
        print(" -", r)
    print("=" * 70)


## Section 9 — Summary

`processed/STATUS.txt` is the single file worth checking before starting Stage 3
(baseline training). If it reads `READY_FOR_TRAINING`, everything under `processed/`
-- shards, metadata, config, channel tables, QC report, split manifests and the
leakage-validation report -- is self-consistent and stamped with the same
`preproc_id`. If it reads `FAILED`, the listed reasons point at exactly which gate
did not pass; re-run only what is needed (a fresh QC pass reads shards already on
disk and does not require re-downloading or re-exporting anything that already
passed).

In [36]:
print(Path(PROCESSED_ROOT, "STATUS.txt").read_text())
def _tree(root, prefix=""):
    entries = sorted(Path(root).iterdir())
    for i, p in enumerate(entries):
        connector = "+-- " if i < len(entries) - 1 else "\\-- "
        print(prefix + connector + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir() and prefix.count("|") < 2:
            _tree(p, prefix + ("|   " if i < len(entries) - 1 else "    "))
_tree(PROCESSED_ROOT)


READY_FOR_TRAINING

+-- STATUS.txt
+-- _export_checkpoint.json
+-- channels/
|   +-- channel_mapping_table.csv
|   +-- channel_masks.npz
|   \-- target_channels.json
+-- config/
|   +-- experiment_config.json
|   \-- preproc_id.txt
+-- data/
|   +-- bnci2014001/
|   |   +-- S001.npz
|   |   +-- S002.npz
|   |   +-- S003.npz
|   |   +-- S004.npz
|   |   +-- S005.npz
|   |   +-- S006.npz
|   |   +-- S007.npz
|   |   +-- S008.npz
|   |   \-- S009.npz
|   +-- cho2017/
|   |   +-- S001.npz
|   |   +-- S002.npz
|   |   +-- S003.npz
|   |   +-- S004.npz
|   |   +-- S005.npz
|   |   +-- S006.npz
|   |   +-- S007.npz
|   |   +-- S008.npz
|   |   +-- S009.npz
|   |   +-- S010.npz
|   |   +-- S011.npz
|   |   +-- S012.npz
|   |   +-- S013.npz
|   |   +-- S014.npz
|   |   +-- S015.npz
|   |   +-- S016.npz
|   |   +-- S017.npz
|   |   +-- S018.npz
|   |   +-- S019.npz
|   |   +-- S020.npz
|   |   +-- S021.npz
|   |   +-- S022.npz
|   |   +-- S023.npz
|   |   +-- S024.npz
|   |   +-- S025.npz
|   | 